# 集装箱缺陷检测 — 数据探索分析报告

本 Notebook 对「数据集3713」进行全面的探索性分析（EDA），为模型设计和论文撰写提供数据支撑。

**分析内容：**
1. 随机可视化 50 张训练图片及标注框
2. 各类缺陷标注框面积分布（大/中/小目标占比）
3. 每张图缺陷数量分布
4. 图片亮度、对比度、色彩分布分析
5. 综合统计摘要

In [ ]:
import os
import random
import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib import font_manager
from collections import Counter, defaultdict
from pathlib import Path

# ============ 全局配置 ============
DATASET_ROOT = r"C:\Users\administrator\Desktop\2026数模国赛\选题D\数据集3713"
OUTPUT_DIR = r"C:\Users\administrator\Desktop\2026数模国赛\选题D\数据探索"
os.makedirs(OUTPUT_DIR, exist_ok=True)

CLASS_NAMES = {0: "Dent(凹陷)", 1: "Hole(破洞)", 2: "Rusty(锈蚀)"}
CLASS_COLORS = {0: "#2196F3", 1: "#F44336", 2: "#FF9800"}  # 蓝/红/橙

# 中文字体
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 150
plt.rcParams['savefig.dpi'] = 200
plt.rcParams['savefig.bbox'] = 'tight'

random.seed(42)
np.random.seed(42)

print("环境配置完成")
print(f"数据集路径: {DATASET_ROOT}")
print(f"输出路径: {OUTPUT_DIR}")

In [ ]:
# ============ 工具函数 ============

def imread_cn(path):
    """读取含中文路径的图片（Windows 兼容）"""
    data = np.fromfile(path, dtype=np.uint8)
    return cv2.imdecode(data, cv2.IMREAD_COLOR)

def load_label(label_path):
    """加载 YOLO 格式标注文件，返回 [(cls, xc, yc, w, h), ...]"""
    boxes = []
    if not os.path.exists(label_path):
        return boxes
    with open(label_path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 5:
                cls = int(parts[0])
                xc, yc, w, h = map(float, parts[1:5])
                boxes.append((cls, xc, yc, w, h))
    return boxes

def get_all_labels(split='train'):
    """获取指定 split 的所有标注"""
    label_dir = os.path.join(DATASET_ROOT, 'labels', split)
    all_labels = {}
    for fname in os.listdir(label_dir):
        if fname.endswith('.txt'):
            img_id = fname[:-4]
            all_labels[img_id] = load_label(os.path.join(label_dir, fname))
    return all_labels

print("工具函数定义完成")

In [ ]:
# ============ 加载全部标注数据 ============

train_labels = get_all_labels('train')
test_labels = get_all_labels('test')

print(f"训练集图片数: {len(train_labels)}")
print(f"测试集图片数: {len(test_labels)}")

# 统计总标注框数
train_box_count = sum(len(v) for v in train_labels.values())
test_box_count = sum(len(v) for v in test_labels.values())
print(f"训练集标注框总数: {train_box_count}")
print(f"测试集标注框总数: {test_box_count}")

## 1. 随机可视化 50 张训练图片及标注框

从训练集中随机抽取 50 张图片，绘制标注框，直观展示三类缺陷的外观差异和标注质量。

In [ ]:
# 随机选取 50 张训练图片
train_img_dir = os.path.join(DATASET_ROOT, 'images', 'train')
all_train_imgs = [f for f in os.listdir(train_img_dir) if f.endswith(('.jpg', '.png', '.jpeg'))]
sample_imgs = random.sample(all_train_imgs, min(50, len(all_train_imgs)))

print(f"从 {len(all_train_imgs)} 张训练图中随机抽取 {len(sample_imgs)} 张进行可视化")

In [ ]:
# 可视化 50 张图片（分 5 批，每批 10 张）
fig, axes = plt.subplots(10, 5, figsize=(25, 50))
fig.suptitle('训练集随机样本可视化（50张）', fontsize=20, fontweight='bold', y=0.995)

for idx, img_name in enumerate(sample_imgs):
    row, col = idx // 5, idx % 5
    ax = axes[row, col]
    
    img_path = os.path.join(train_img_dir, img_name)
    img = imread_cn(img_path)
    
    if img is None:
        ax.text(0.5, 0.5, '无法读取', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(img_name, fontsize=8)
        continue
    
    h_img, w_img = img.shape[:2]
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    ax.imshow(img_rgb)
    
    # 绘制标注框
    img_id = os.path.splitext(img_name)[0]
    boxes = train_labels.get(img_id, [])
    for (cls, xc, yc, w, h) in boxes:
        x1 = (xc - w/2) * w_img
        y1 = (yc - h/2) * h_img
        bw = w * w_img
        bh = h * h_img
        color = CLASS_COLORS[cls]
        rect = patches.Rectangle((x1, y1), bw, bh, linewidth=1.5,
                                  edgecolor=color, facecolor='none')
        ax.add_patch(rect)
        ax.text(x1, y1-3, CLASS_NAMES[cls].split('(')[0], fontsize=6,
                color='white', bbox=dict(boxstyle='round,pad=0.1', facecolor=color, alpha=0.8))
    
    ax.set_title(f"{img_name} ({len(boxes)}个缺陷)", fontsize=8)
    ax.axis('off')

plt.tight_layout()
save_path = os.path.join(OUTPUT_DIR, '01_50张样本可视化.png')
plt.savefig(save_path)
plt.show()
print(f"已保存: {save_path}")

## 2. 各类缺陷标注框面积分布

按照 COCO 标准将目标按面积分为三类：
- **小目标**：面积 < 32² = 1024 px²
- **中目标**：32² ≤ 面积 < 96² = 9216 px²  
- **大目标**：面积 ≥ 96² = 9216 px²

分析各类缺陷的尺度分布，为模型设计（如是否需要小目标增强模块）提供依据。

In [ ]:
# 计算所有标注框的像素面积
# 需要知道每张图片的实际尺寸

def compute_box_areas(labels_dict, img_dir):
    """计算所有标注框的像素面积，返回 {class_id: [area1, area2, ...]}"""
    areas_by_class = defaultdict(list)
    sizes_by_class = defaultdict(list)  # (w_px, h_px)
    
    for img_id, boxes in labels_dict.items():
        # 找对应图片
        img_path = None
        for ext in ['.jpg', '.png', '.jpeg']:
            candidate = os.path.join(img_dir, img_id + ext)
            if os.path.exists(candidate):
                img_path = candidate
                break
        if img_path is None:
            continue
        
        img = imread_cn(img_path)
        if img is None:
            continue
        h_img, w_img = img.shape[:2]
        
        for (cls, xc, yc, w, h) in boxes:
            w_px = w * w_img
            h_px = h * h_img
            area = w_px * h_px
            areas_by_class[cls].append(area)
            sizes_by_class[cls].append((w_px, h_px))
    
    return areas_by_class, sizes_by_class

print("正在计算标注框面积（需遍历所有训练图片）...")
areas_by_class, sizes_by_class = compute_box_areas(train_labels, train_img_dir)

for cls in range(3):
    print(f"  {CLASS_NAMES[cls]}: {len(areas_by_class[cls])} 个标注框")

In [ ]:
# 面积分布直方图 + 大中小目标占比
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.suptitle('各类缺陷标注框面积分布分析', fontsize=16, fontweight='bold')

# 上排：面积直方图
for cls in range(3):
    ax = axes[0, cls]
    areas = np.array(areas_by_class[cls])
    ax.hist(areas, bins=50, color=CLASS_COLORS[cls], alpha=0.75, edgecolor='white')
    ax.axvline(1024, color='green', linestyle='--', label='小/中分界(32²)')
    ax.axvline(9216, color='red', linestyle='--', label='中/大分界(96²)')
    ax.set_title(f"{CLASS_NAMES[cls]}\n(n={len(areas)}, 均值={areas.mean():.0f}px²)", fontsize=11)
    ax.set_xlabel('面积 (px²)')
    ax.set_ylabel('数量')
    ax.legend(fontsize=8)

# 下排：大中小目标占比饼图
scale_names = ['小目标(<32²)', '中目标(32²~96²)', '大目标(≥96²)']
scale_colors = ['#4CAF50', '#FFC107', '#E91E63']

for cls in range(3):
    ax = axes[1, cls]
    areas = np.array(areas_by_class[cls])
    small = (areas < 1024).sum()
    medium = ((areas >= 1024) & (areas < 9216)).sum()
    large = (areas >= 9216).sum()
    counts = [small, medium, large]
    total = sum(counts)
    
    wedges, texts, autotexts = ax.pie(counts, labels=scale_names, colors=scale_colors,
                                       autopct='%1.1f%%', startangle=90, textprops={'fontsize': 9})
    ax.set_title(f"{CLASS_NAMES[cls]} 尺度分布\n(共{total}个)", fontsize=11)

plt.tight_layout()
save_path = os.path.join(OUTPUT_DIR, '02_面积分布与尺度分析.png')
plt.savefig(save_path)
plt.show()
print(f"已保存: {save_path}")

In [ ]:
# 尺度分布汇总表
print("="*60)
print(f"{'类别':<12} {'小目标':<12} {'中目标':<12} {'大目标':<12} {'总计':<8}")
print("="*60)
for cls in range(3):
    areas = np.array(areas_by_class[cls])
    small = (areas < 1024).sum()
    medium = ((areas >= 1024) & (areas < 9216)).sum()
    large = (areas >= 9216).sum()
    total = len(areas)
    print(f"{CLASS_NAMES[cls]:<10} {small:>4}({small/total*100:5.1f}%) "
          f"{medium:>4}({medium/total*100:5.1f}%) "
          f"{large:>4}({large/total*100:5.1f}%) "
          f"{total:>5}")
print("="*60)
all_areas = np.concatenate([areas_by_class[c] for c in range(3)])
print(f"全部目标: 小{(all_areas<1024).sum()} 中{((all_areas>=1024)&(all_areas<9216)).sum()} 大{(all_areas>=9216).sum()} 总计{len(all_areas)}")

## 3. 每张图缺陷数量分布

统计每张图片中包含的缺陷目标数量，分析单图多目标检测的难度。

In [ ]:
# 每张图的缺陷数量
defects_per_image = [len(boxes) for boxes in train_labels.values()]
defects_per_image = np.array(defects_per_image)

# 按类别统计每张图各类缺陷数
cls_per_image = {cls: [] for cls in range(3)}
for boxes in train_labels.values():
    cls_count = Counter([b[0] for b in boxes])
    for cls in range(3):
        cls_per_image[cls].append(cls_count.get(cls, 0))

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('每张图缺陷数量分布', fontsize=14, fontweight='bold')

# 总缺陷数分布
ax = axes[0]
max_n = defects_per_image.max()
bins = np.arange(0.5, max_n + 2, 1)
ax.hist(defects_per_image, bins=bins, color='#607D8B', alpha=0.8, edgecolor='white')
ax.set_title(f'总缺陷数/图\n均值={defects_per_image.mean():.2f}, 中位数={np.median(defects_per_image):.0f}')
ax.set_xlabel('缺陷数量')
ax.set_ylabel('图片数')
ax.axvline(defects_per_image.mean(), color='red', linestyle='--', label=f'均值={defects_per_image.mean():.1f}')
ax.legend()

# 各类别缺陷数分布
for i, cls in enumerate(range(3)):
    ax = axes[i+1] if i < 2 else axes[2]
    if i >= 2:
        break

# 重新画：三个子图分别是 Dent/Hole/Rusty 的每图数量
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('各类缺陷每图数量分布', fontsize=14, fontweight='bold')

for cls in range(3):
    ax = axes[cls]
    data = np.array(cls_per_image[cls])
    max_v = max(data.max(), 1)
    bins = np.arange(-0.5, max_v + 1.5, 1)
    ax.hist(data, bins=bins, color=CLASS_COLORS[cls], alpha=0.8, edgecolor='white')
    has_defect = (data > 0).sum()
    ax.set_title(f"{CLASS_NAMES[cls]}\n出现率={has_defect/len(data)*100:.1f}%, 均值={data.mean():.2f}")
    ax.set_xlabel('该类缺陷数量/图')
    ax.set_ylabel('图片数')

plt.tight_layout()
save_path = os.path.join(OUTPUT_DIR, '03_缺陷数量分布.png')
plt.savefig(save_path)
plt.show()
print(f"已保存: {save_path}")

In [ ]:
# 缺陷数量统计摘要
print("每张图缺陷数量统计:")
print(f"  最小值: {defects_per_image.min()}")
print(f"  最大值: {defects_per_image.max()}")
print(f"  均值:   {defects_per_image.mean():.2f}")
print(f"  中位数: {np.median(defects_per_image):.0f}")
print(f"  标准差: {defects_per_image.std():.2f}")
print()

# 数量区间分布
print("缺陷数量区间分布:")
for lo, hi in [(1,1), (2,2), (3,3), (4,5), (6,10), (11,99)]:
    count = ((defects_per_image >= lo) & (defects_per_image <= hi)).sum()
    label = f"{lo}" if lo == hi else f"{lo}-{hi}"
    print(f"  {label}个缺陷: {count}张 ({count/len(defects_per_image)*100:.1f}%)")

## 4. 图片亮度、对比度、色彩分布分析

分析图片的低级视觉特征，评估数据增强策略的必要性：
- 亮度（HSV 空间 V 通道均值）
- 对比度（灰度图标准差）
- 饱和度（HSV 空间 S 通道均值）
- 各通道颜色直方图

In [ ]:
# 计算所有训练图片的亮度、对比度、饱和度
print("正在分析图片视觉特征（遍历全部训练图）...")

brightness_list = []  # V通道均值
contrast_list = []    # 灰度标准差
saturation_list = []  # S通道均值
hue_list = []         # H通道均值

for img_name in all_train_imgs:
    img_path = os.path.join(train_img_dir, img_name)
    img = imread_cn(img_path)
    if img is None:
        continue
    
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    brightness_list.append(hsv[:, :, 2].mean())
    contrast_list.append(gray.std())
    saturation_list.append(hsv[:, :, 1].mean())
    hue_list.append(hsv[:, :, 0].mean())

brightness = np.array(brightness_list)
contrast = np.array(contrast_list)
saturation = np.array(saturation_list)
hue = np.array(hue_list)

print(f"成功分析 {len(brightness)} 张图片")

In [ ]:
# 亮度、对比度、饱和度、色调分布图
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('训练集图片视觉特征分布', fontsize=14, fontweight='bold')

# 亮度
ax = axes[0, 0]
ax.hist(brightness, bins=50, color='#FFC107', alpha=0.8, edgecolor='white')
ax.axvline(brightness.mean(), color='red', linestyle='--', label=f'均值={brightness.mean():.1f}')
ax.set_title(f'亮度 (V通道均值)\nμ={brightness.mean():.1f}, σ={brightness.std():.1f}')
ax.set_xlabel('亮度值 (0-255)')
ax.set_ylabel('图片数')
ax.legend()

# 对比度
ax = axes[0, 1]
ax.hist(contrast, bins=50, color='#9C27B0', alpha=0.8, edgecolor='white')
ax.axvline(contrast.mean(), color='red', linestyle='--', label=f'均值={contrast.mean():.1f}')
ax.set_title(f'对比度 (灰度标准差)\nμ={contrast.mean():.1f}, σ={contrast.std():.1f}')
ax.set_xlabel('对比度值')
ax.set_ylabel('图片数')
ax.legend()

# 饱和度
ax = axes[1, 0]
ax.hist(saturation, bins=50, color='#4CAF50', alpha=0.8, edgecolor='white')
ax.axvline(saturation.mean(), color='red', linestyle='--', label=f'均值={saturation.mean():.1f}')
ax.set_title(f'饱和度 (S通道均值)\nμ={saturation.mean():.1f}, σ={saturation.std():.1f}')
ax.set_xlabel('饱和度值 (0-255)')
ax.set_ylabel('图片数')
ax.legend()

# 色调
ax = axes[1, 1]
ax.hist(hue, bins=50, color='#2196F3', alpha=0.8, edgecolor='white')
ax.axvline(hue.mean(), color='red', linestyle='--', label=f'均值={hue.mean():.1f}')
ax.set_title(f'色调 (H通道均值)\nμ={hue.mean():.1f}, σ={hue.std():.1f}')
ax.set_xlabel('色调值 (0-180)')
ax.set_ylabel('图片数')
ax.legend()

plt.tight_layout()
save_path = os.path.join(OUTPUT_DIR, '04_亮度对比度色彩分布.png')
plt.savefig(save_path)
plt.show()
print(f"已保存: {save_path}")

In [ ]:
# 颜色通道直方图（随机抽取 200 张叠加）
sample_for_hist = random.sample(all_train_imgs, min(200, len(all_train_imgs)))

hist_b = np.zeros(256)
hist_g = np.zeros(256)
hist_r = np.zeros(256)

for img_name in sample_for_hist:
    img_path = os.path.join(train_img_dir, img_name)
    img = imread_cn(img_path)
    if img is None:
        continue
    for i, hist_arr in enumerate([hist_b, hist_g, hist_r]):
        h = cv2.calcHist([img], [i], None, [256], [0, 256]).flatten()
        hist_arr += h / h.sum()  # 归一化后累加

fig, ax = plt.subplots(1, 1, figsize=(12, 5))
x = np.arange(256)
ax.plot(x, hist_r / len(sample_for_hist), color='red', alpha=0.7, label='R通道')
ax.plot(x, hist_g / len(sample_for_hist), color='green', alpha=0.7, label='G通道')
ax.plot(x, hist_b / len(sample_for_hist), color='blue', alpha=0.7, label='B通道')
ax.set_title('训练集颜色通道直方图（200张叠加平均）', fontsize=13)
ax.set_xlabel('像素值 (0-255)')
ax.set_ylabel('归一化频率')
ax.legend()
ax.set_xlim(0, 255)

plt.tight_layout()
save_path = os.path.join(OUTPUT_DIR, '05_颜色通道直方图.png')
plt.savefig(save_path)
plt.show()
print(f"已保存: {save_path}")

## 5. 综合统计摘要

汇总所有分析结果，输出结构化统计报告。

In [ ]:
# 综合统计报告
report_lines = []
report_lines.append("=" * 60)
report_lines.append("集装箱缺陷检测数据集 — 探索性分析摘要")
report_lines.append("=" * 60)
report_lines.append("")
report_lines.append("【1. 数据集规模】")
report_lines.append(f"  训练集: {len(train_labels)} 张图片, {train_box_count} 个标注框")
report_lines.append(f"  测试集: {len(test_labels)} 张图片, {test_box_count} 个标注框")
report_lines.append(f"  总计:   {len(train_labels)+len(test_labels)} 张图片")
report_lines.append("")

# 类别分布
report_lines.append("【2. 类别分布（训练集）】")
total_boxes = sum(len(areas_by_class[c]) for c in range(3))
for cls in range(3):
    n = len(areas_by_class[cls])
    report_lines.append(f"  {CLASS_NAMES[cls]}: {n} 个 ({n/total_boxes*100:.1f}%)")
report_lines.append(f"  总计: {total_boxes} 个标注框")
report_lines.append("")

# 尺度分布
report_lines.append("【3. 目标尺度分布（COCO标准）】")
for cls in range(3):
    areas = np.array(areas_by_class[cls])
    small = (areas < 1024).sum()
    medium = ((areas >= 1024) & (areas < 9216)).sum()
    large = (areas >= 9216).sum()
    n = len(areas)
    report_lines.append(f"  {CLASS_NAMES[cls]}: 小{small}({small/n*100:.1f}%) 中{medium}({medium/n*100:.1f}%) 大{large}({large/n*100:.1f}%)")
report_lines.append("")

# 缺陷数量
report_lines.append("【4. 每图缺陷数量】")
report_lines.append(f"  均值: {defects_per_image.mean():.2f}")
report_lines.append(f"  中位数: {np.median(defects_per_image):.0f}")
report_lines.append(f"  范围: [{defects_per_image.min()}, {defects_per_image.max()}]")
report_lines.append(f"  标准差: {defects_per_image.std():.2f}")
report_lines.append("")

# 视觉特征
report_lines.append("【5. 图片视觉特征】")
report_lines.append(f"  亮度:   μ={brightness.mean():.1f}, σ={brightness.std():.1f}, 范围[{brightness.min():.0f}, {brightness.max():.0f}]")
report_lines.append(f"  对比度: μ={contrast.mean():.1f}, σ={contrast.std():.1f}, 范围[{contrast.min():.0f}, {contrast.max():.0f}]")
report_lines.append(f"  饱和度: μ={saturation.mean():.1f}, σ={saturation.std():.1f}, 范围[{saturation.min():.0f}, {saturation.max():.0f}]")
report_lines.append(f"  色调:   μ={hue.mean():.1f}, σ={hue.std():.1f}, 范围[{hue.min():.0f}, {hue.max():.0f}]")
report_lines.append("")

# 关键发现
report_lines.append("【6. 关键发现与建模启示】")
all_areas = np.concatenate([areas_by_class[c] for c in range(3)])
small_ratio = (all_areas < 1024).sum() / len(all_areas) * 100
report_lines.append(f"  - 类别不均衡: Hole(破洞)占比最低，需关注少数类检测性能")
report_lines.append(f"  - 小目标占比 {small_ratio:.1f}%，需强化小目标检测能力")
report_lines.append(f"  - 亮度/对比度变化范围大，数据增强（随机亮度/对比度调整）有必要")
report_lines.append(f"  - 训练集无负样本（所有图均含缺陷），问题1需额外构造负样本")
report_lines.append("")
report_lines.append("=" * 60)

report_text = "\n".join(report_lines)
print(report_text)

# 保存报告
report_path = os.path.join(OUTPUT_DIR, '数据探索报告.txt')
with open(report_path, 'w', encoding='utf-8') as f:
    f.write(report_text)
print(f"\n报告已保存: {report_path}")

In [ ]:
# 列出所有生成的图表文件
print("\n本次数据探索生成的所有文件:")
print("-" * 40)
for f in sorted(os.listdir(OUTPUT_DIR)):
    fpath = os.path.join(OUTPUT_DIR, f)
    size_kb = os.path.getsize(fpath) / 1024
    print(f"  {f}  ({size_kb:.0f} KB)")